In [1]:
import joblib
import pandas as pd
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load best model và vectorizer
model = joblib.load('best_xgboost_model.pkl')
vectorizer = joblib.load('tfidf_vectorizer.pkl')

# Load tập Test
X_test = sp.load_npz('X_test_tfidf.npz')
y_test = pd.read_csv('y_test.csv').values.ravel()

LABELS = ['negative', 'neutral', 'positive']

print(" Đã load model và tập Test thành công!")
print(f"Test set size: {X_test.shape[0]} samples")

 Đã load model và tập Test thành công!
Test set size: 2196 samples


In [2]:
# Dự đoán trên tập Test
y_pred_test = model.predict(X_test)

# Tính các chỉ số
acc = accuracy_score(y_test, y_pred_test)
f1w = f1_score(y_test, y_pred_test, average='weighted')
f1m = f1_score(y_test, y_pred_test, average='macro')

print("=" * 60)
print("FINAL EVALUATION - TRÊN TẬP TEST")
print("=" * 60)
print(f"Accuracy          : {acc:.4f}")
print(f"F1 Weighted       : {f1w:.4f}")
print(f"F1 Macro          : {f1m:.4f}")
print("=" * 60)

print("\nClassification Report on Test Set:")
print(classification_report(y_test, y_pred_test, target_names=LABELS))

FINAL EVALUATION - TRÊN TẬP TEST
Accuracy          : 0.7869
F1 Weighted       : 0.7826
F1 Macro          : 0.7210

Classification Report on Test Set:
              precision    recall  f1-score   support

    negative       0.84      0.89      0.86      1377
     neutral       0.64      0.58      0.61       465
    positive       0.73      0.64      0.68       354

    accuracy                           0.79      2196
   macro avg       0.74      0.71      0.72      2196
weighted avg       0.78      0.79      0.78      2196



In [7]:
# ==================== CELL 3: SCRAPE TẤT CẢ REVIEW TỪ SKYTRAX ====================

import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd

def clean_text_for_prediction(text):
    """Làm sạch văn bản - Không giới hạn số từ"""
    if not text:
        return ""
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

def predict_sentiment(text):
    X = vectorizer.transform([text])
    pred = model.predict(X)[0]
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    return label_map[pred]

def scrape_all_skytrax_reviews(base_url, max_pages=20):
    """
    Scrape TẤT CẢ review từ Skytrax bằng cách duyệt đúng pagination
    """
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    results = []
    page = 1
    
    print(f"🌐 Bắt đầu scrape tất cả review từ Uzbekistan Airways...\n")
    
    while page <= max_pages:
        # Xây dựng URL đúng cho từng trang
        if page == 1:
            url = base_url
        else:
            url = base_url.rstrip('/') + f"/page/{page}/"
        
        try:
            print(f"Đang lấy trang {page} → {url}")
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code == 404:
                print(f"Trang {page} không tồn tại. Kết thúc.")
                break
            if response.status_code != 200:
                print(f"Lỗi status code {response.status_code} ở trang {page}")
                break
                
            soup = BeautifulSoup(response.text, 'html.parser')
            review_blocks = soup.find_all('article', itemprop='review')
            
            if not review_blocks:
                print("Không còn review nào.")
                break
            
            print(f"   → Tìm thấy {len(review_blocks)} review trên trang {page}")
            
            for review in review_blocks:
                text_tag = review.find('div', class_='text_content')
                if not text_tag:
                    continue
                    
                raw_text = text_tag.get_text(strip=True)
                clean_text = clean_text_for_prediction(raw_text)
                
                if len(clean_text.split()) < 8:
                    continue
                    
                sentiment = predict_sentiment(clean_text)
                
                results.append({
                    'page': page,
                    'raw_text': raw_text,
                    'clean_text': clean_text,
                    'predicted_sentiment': sentiment
                })
                
            page += 1
            time.sleep(1.0)  # Nghỉ 1 giây giữa các trang
            
        except Exception as e:
            print(f"Lỗi ở trang {page}: {e}")
            break
    
    # === LƯU KẾT QUẢ ===
    df = pd.DataFrame(results)
    df.to_csv('skytrax_uzbekistan_all_predictions.csv', index=False, encoding='utf-8')
    
    # Thống kê
    total = len(df)
    if total > 0:
        stats = df['predicted_sentiment'].value_counts()
        print("\n" + "="*65)
        print(f"HOÀN TẤT - TỔNG CỘNG {total} REVIEW ĐÃ ĐƯỢC PHÂN TÍCH")
        print("="*65)
        for sent, count in stats.items():
            print(f"{sent.capitalize():8}: {count:3d} reviews ({count/total*100:5.1f}%)")
    
    print(f"\n✅ Đã lưu tất cả kết quả vào file: **skytrax_uzbekistan_all_predictions.csv**")
    return df

# ====================== CHẠY ======================

airline_url = "https://www.airlinequality.com/airline-reviews/uzbekistan-airways"

df_results = scrape_all_skytrax_reviews(airline_url, max_pages=20)

🌐 Bắt đầu scrape tất cả review từ Uzbekistan Airways...

Đang lấy trang 1 → https://www.airlinequality.com/airline-reviews/uzbekistan-airways
   → Tìm thấy 10 review trên trang 1
Đang lấy trang 2 → https://www.airlinequality.com/airline-reviews/uzbekistan-airways/page/2/
   → Tìm thấy 10 review trên trang 2
Đang lấy trang 3 → https://www.airlinequality.com/airline-reviews/uzbekistan-airways/page/3/
   → Tìm thấy 10 review trên trang 3
Đang lấy trang 4 → https://www.airlinequality.com/airline-reviews/uzbekistan-airways/page/4/
   → Tìm thấy 10 review trên trang 4
Đang lấy trang 5 → https://www.airlinequality.com/airline-reviews/uzbekistan-airways/page/5/
   → Tìm thấy 10 review trên trang 5
Đang lấy trang 6 → https://www.airlinequality.com/airline-reviews/uzbekistan-airways/page/6/
   → Tìm thấy 10 review trên trang 6
Đang lấy trang 7 → https://www.airlinequality.com/airline-reviews/uzbekistan-airways/page/7/
   → Tìm thấy 10 review trên trang 7
Đang lấy trang 8 → https://www.airlinequa

In [8]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv('skytrax_uzbekistan_all_predictions.csv')
print("\n Phân bố sentiment:")
print(df['predicted_sentiment'].value_counts())

# Nếu muốn xem chi tiết hơn
print("\n Chi tiết đầy đủ:")
print(df.head(10))


 Phân bố sentiment:
predicted_sentiment
negative    69
positive    14
Name: count, dtype: int64

 Chi tiết đầy đủ:
   page                                           raw_text  \
0     1  ✅Trip Verified|   We travelled as a family fro...   
1     1  ✅Trip Verified|  We checked in when check in o...   
2     1  ✅Trip Verified|   Overall the flight was good,...   
3     1  ✅Trip Verified|   The Mineralnye Vody to Tashk...   
4     1  Not Verified|  My experience with Uzbekistan A...   
5     1  ✅Trip Verified| This is by far the worst airli...   
6     1  ✅Trip Verified|   Checked in online but was un...   
7     1  ✅Trip Verified|  By far the worst airline! I f...   
8     1  ✅Trip Verified| As a well travelled Belgian wh...   
9     1  ✅Trip Verified| Check-in prolonged because the...   

                                          clean_text predicted_sentiment  
0  trip verified we travelled as a family from mi...            positive  
1  trip verified we checked in when check in open..